# PascalVOC-SP: readout asymmetry and edge co-training

Same node vs node+boundary readout on **CPEN** (linear incidence) and **CAPEN-Llama** (`incidence_m22`, 4 GPU, effective batch $16\times 4$). The trunk differs; the readout intervention is the same.

Always score **node macro-F1** as the primary metric. Plots also show val/test accuracy and train loss. The edge loss is an intervention, not the reported task.


## Trunk (CPEN)

Linear encoders, $\mW_X^{(0)}\in\mathbb{R}^{n_0\times D}$, $\mW_E^{(0)}\in\mathbb{R}^{m_0\times D}$, iid $\mathcal{N}(0,D^{-1})$:

$$
\mX^{(0)}=\sqrt{\frac{D}{n_0}}\,\mX\mW_X^{(0)},
\qquad
\mE^{(0)}=\sqrt{\frac{D}{m_0}}\,\mE\mW_E^{(0)}.
$$

PascalVOC-SP: $n_0=14$, $m_0=2$. Residual structure is **transformer-like** (operator mix, then MLP). Particle and edge updates use the **same** $\mX^{(\ell)},\mE^{(\ell)}$. No dropout.

$$
\begin{aligned}
\tilde{\mE}^{(\ell+1)}
&=\mE^{(\ell)}+\frac{1}{\sqrt{2}L}\Big[
f_{22}^{(\ell+1)}(\mE^{(\ell)},\mE^{(\ell)})
+f_{12}^{(\ell+1)}(\mE^{(\ell)},\mX^{(\ell)})
\Big],\\
\mE^{(\ell+1)}
&=\tilde{\mE}^{(\ell+1)}+\frac{1}{L}\,\mathrm{MLP}_2^{(\ell+1)}(\tilde{\mE}^{(\ell+1)}),\\
\tilde{\mX}^{(\ell+1)}
&=\mX^{(\ell)}+\frac{1}{\sqrt{2}L}\Big[
f_{11}^{(\ell+1)}(\mX^{(\ell)},\mX^{(\ell)})
+f_{21}^{(\ell+1)}(\mX^{(\ell)},\mE^{(\ell)})
\Big],\\
\mX^{(\ell+1)}
&=\tilde{\mX}^{(\ell+1)}+\frac{1}{L}\,\mathrm{MLP}_1^{(\ell+1)}(\tilde{\mX}^{(\ell+1)}).
\end{aligned}
$$

MLPs (GELU $\phi$, LN with no affine params, $\epsilon=10^{-6}$):

$$
\mathrm{MLP}(\mZ)=\frac{1}{\sqrt{4D}}\,\phi\!\left(\frac{1}{\sqrt{D}}\,\overline{\mZ}\,\mW_1\right)\mW_2,
\quad
\overline{\mZ}=\mathrm{LN}(\mZ),
$$

with $\mW_1\in\mathbb{R}^{D\times 4D}$, $\mW_2\in\mathbb{R}^{4D\times D}$ iid $\mathcal{N}(0,1)$.

### Operators $f_{rs}$

Let $S\in\{0,1\}^{M\times N}$ be the (padded) incidence matrix of the native undirected Pascal graph, and $A\in\{0,1\}^{N\times N}$ the corresponding **node adjacency** (no self-loops). Message passing is linear in the second argument:

| $f_{rs}$ | map | operator |
|---|---|---|
| $f_{11}(X,X)$ | $N\to N$ | **adjacency** $A$ |
| $f_{12}(E,X)$ | $N\to M$ | **incidence** $S$ |
| $f_{21}(X,E)$ | $M\to N$ | **$S^\top$** |
| $f_{22}(E,E)$ | $M\to M$ | **$SS^\top$** |

Degree-normalized as in CPEN (`operator_normalization=degree`):

$$
T_{12}=D_E^{-1}S,\quad
T_{21}=D_X^{-1}S^\top,\quad
T_{22}=D_E^{-1}S\,D_X^{-1}S^\top,\quad
T_{11}=D_A^{-1/2}(A+I)D_A^{-1/2}.
$$

$T_{11}$ is **$A$**, not $S^\top S$. On 2-edges those differ by the degree on the diagonal.


## Readout: graph CPEN vs this experiment

Graph-level CPEN (jets) scores **both** channels with the same logit,

$$
z_X=\frac{1}{N\sqrt{D}}\sum_{u=1}^N \mX_u^{(L)}\cdot w_X,
\qquad
z_E=\frac{1}{M\sqrt{D}}\sum_{e=1}^M \mE_e^{(L)}\cdot w_E,
\qquad
z=\frac{1}{\sqrt{2}}(z_X+z_E),
$$

$w_X,w_E$ iid $\mathcal{N}(0,D^{-1})$. That is **not** used here.

Stock CPEN **node** mode is also not used as-is: it mixes class logits as $(z_X + T_{21} z_E)/\sqrt{2}$ with $W_E\in\mathbb{R}^{D\times 21}$. This study turns that off (`x_only_node_readout`) so the node task is $W_X$ on $X^{(L)}$ only.

Those pooled $z_X,z_E$ are **not** used here. The only readouts are the same two decoders, applied to the **node features** and the **edge features** (no extra classifier):

$$
\mathrm{logits}_u
=
\frac{1}{\sqrt{D}}\,
\mX_u^{(L)}\,W_X
\qquad
\mathrm{logits}_e
=
\frac{1}{\sqrt{D}}\,
\mE_e^{(L)}\,W_E,
$$

$W_X\in\mathbb{R}^{D\times 21}$, $W_E\in\mathbb{R}^{D\times 2}$, columns iid $\mathcal{N}(0,D^{-1})$ (same as $w_X,w_E$ in the graph decoder, one column per class). $21$ is Pascal’s node-class count; $2$ is interior vs boundary.

- **node-only:** train $W_X$ with node CE. $W_E$ is unused (no edge scores).
- **node + boundary:** train both. Node CE on $\mathrm{logits}_u$; boundary CE on $\mathrm{logits}_e$ with $y_e=\mathbf{1}[y_u\neq y_v]$.

The hypothesis is that dropping the edge readout is why node-cls LR-transfer is weaker, and that scoring $E$ with $W_E$ restores the $z_E$ channel.


## Edge co-training task

PascalVOC-SP: SLIC superpixels, native 2-edges, 21 node classes. Val `incidence-v2`:

| quantity | value |
|---|---|
| graphs | 1428 |
| nodes | $6.85\times 10^5$ (class 0 $\approx 70\%$) |
| undirected edges | $1.94\times 10^6$ |
| $y_u=y_v$ (interior) | **91.8%** |
| $y_u\neq y_v$ (boundary) | **8.2%** |
| $\cos(x_u,x_v)$ same / diff | $0.997$ / $0.998$ |

Do **not** co-train on input-feature similarity: adjacent superpixels are already identical in the 14-D features.

**Primary:** hard boundary $y_e=\mathbf{1}[y_u\neq y_v]$, binary CE, pos-weight $\approx 11$. Same-class is a 92% majority predictor. No 21-class (or $21\times 21$) scores on edges.

**Secondary (later):** product-consistency $\langle\mathrm{softmax}(z_u),\mathrm{softmax}(z_v)\rangle$ with stopgrad, as on streams.

**Control (later):** shuffled $y_e$. If that still “fixes” the LR curve, the effect is extra regularization of $E$, not readout alignment.


## Ablation grid (short run)

Two families, identical trunk:

1. **node-only** — decode $X^{(L)}$ only (`readout_mode=node`).
2. **node + boundary** — decode $X^{(L)}$ and $E^{(L)}$ (`readout_mode=node+edge`).

| knob | choice |
|---|---|
| model | **CPEN** |
| $f_{11}$ | adjacency $A$ |
| $f_{12},f_{21},f_{22}$ | $S$, $S^\top$, $SS^\top$ |
| residual | transformer-like |
| LN / dropout | yes / **none** |
| $L$ | 4 |
| $D$ | $\{64,128,256\}$ |
| $\eta_0$ | $\{0.05,0.1,0.25,0.5,1.0\}$, $\eta=\eta_0/\sqrt{D}$ |
| $\lambda$ | $1$ |
| seed | $\{0,1,2\}$ (in the run name as `seed{N}`) |
| epochs | **5** (extend once the $\eta_0$ peak is visible) |
| val | every epoch, full val |
| metric | `val_f1` (macro, nodes) vs $\eta_0$, mean $\pm$ std over seeds |

Hypothesis: node-only LR-scan peaks **drift or flatten** as $D$ grows; boundary co-training **locks $\eta_0^\star$** the way $z_X+z_E$ does on jets.

Labels $y_e$ from incidence + node $y$ at train time (no cache rebuild). Log edge-AUROC only as a check that the aux is learnable.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import torch

REPO = Path("../..").resolve()
sys.path.insert(0, str(REPO / "src"))

CACHE = Path(
    "/scratch/gpfs/BHANIN/jdezoort/datasets/processed/pascalvoc-sp/incidence-v2/val.pt"
)
print("cache:", CACHE, "exists" if CACHE.is_file() else "MISSING")

In [ ]:
def edge_endpoint_labels(payload: dict, graph: int) -> tuple[torch.Tensor, torch.Tensor]:
    m = int(payload["n_edges"][graph])
    nodes = payload["incidence_node"][graph, : 2 * m].to(torch.long)
    src, dst = nodes[0::2], nodes[1::2]
    y = payload["y"][graph]
    return y[src], y[dst]


payload = torch.load(CACHE, map_location="cpu", weights_only=False)
same = tot = 0
for g in range(int(payload["y"].size(0))):
    ys, yd = edge_endpoint_labels(payload, g)
    same += int((ys == yd).sum())
    tot += int(ys.numel())
print(f"val edges={tot}  same-class={same/tot:.3f}  boundary={(tot-same)/tot:.3f}")
print(f"pos_weight for boundary CE ≈ {same / max(tot - same, 1):.2f}")

## Jobs (seed 0, Adam)

Hardcoded Slurm array tasks. Parquets live under `pascal_output/adam/sweep_lr`; logs in `scans/pascal/slurm-<job>_0.out`.

### CPEN (1 GPU)

| family | $L,D$ | $0.01$ | $0.05$ | $0.25$ | $1$ | $2.5$ |
|---|---|---|---|---|---|---|
| node | $4,256$ | 13114821 | 13108447 | 13101785 | 13108417 | 13114860 |
| node | $6,512$ | 13113750 | 13111695 | 13108772 | 13110652 | 13113772 |
| node+edge | $4,256$ | — | 13108440 | 13101408 | 13102930 | — |
| node+edge | $6,512$ | 13113735 | 13110672 | 13108758 | 13110655 | 13113754 |

### CAPEN-Llama (`incm22`, bs $16\times 4$)

| family | $L,D$ | $0.01$ | $0.05$ | $0.25$ | $1$ | $2.5$ |
|---|---|---|---|---|---|---|
| node | $4,256$ | 13127462 | 13127470 | 13127473 | 13127474 | 13127475 |
| node | $6,512$ | 13127498 | 13127499 | 13127505 | 13127506 | 13127509 |
| node+edge | $4,256$ | 13127227 | 13127235 | 13127236 | 13127237 | 13127239 |
| node+edge | $6,512$ | 13127213 | 13126479 | 13126471 | 13126535 | 13127199 |

Each η0 figure is **CPEN left, CAPEN-Llama right**. Color is $(L,D)$; solid / circles = node, dashed / squares = node+edge. Incomplete jobs still contribute best-so-far val metrics.


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd

REPO = Path("../..").resolve()
sys.path.insert(0, str(REPO / "src"))

import cpen.utils.transfer_plots as transfer_plots

importlib.reload(transfer_plots)

from cpen.utils.transfer_plots import (
    JobSpec,
    PASCAL_OUT_ROOT,
    PASCAL_SLURM_DIRS,
    TransferStudy,
    setup_mpl_style,
)

setup_mpl_style()

CPEN_JOBS = [
    JobSpec("13114821_0", "node", r"η0=0.01, L=4, D=256, seed=0"),
    JobSpec("13108447_0", "node", r"η0=0.05, L=4, D=256, seed=0"),
    JobSpec("13101785_0", "node", r"η0=0.25, L=4, D=256, seed=0"),
    JobSpec("13108417_0", "node", r"η0=1, L=4, D=256, seed=0"),
    JobSpec("13114860_0", "node", r"η0=2.5, L=4, D=256, seed=0"),
    #
    JobSpec("13113750_0", "node", r"η0=0.01, L=6, D=512, seed=0"),
    JobSpec("13111695_0", "node", r"η0=0.05, L=6, D=512, seed=0"),
    JobSpec("13108772_0", "node", r"η0=0.25, L=6, D=512, seed=0"),
    JobSpec("13110652_0", "node", r"η0=1, L=6, D=512, seed=0"),
    JobSpec("13113772_0", "node", r"η0=2.5, L=6, D=512, seed=0"),
    #
    JobSpec("13108440_0", "node+edge", r"η0=0.05, L=4, D=256, seed=0"),
    JobSpec("13101408_0", "node+edge", r"η0=0.25, L=4, D=256, seed=0"),
    JobSpec("13102930_0", "node+edge", r"η0=1, L=4, D=256, seed=0"),
    #
    JobSpec("13113735_0", "node+edge", r"η0=0.01, L=6, D=512, seed=0"),
    JobSpec("13110672_0", "node+edge", r"η0=0.05, L=6, D=512, seed=0"),
    JobSpec("13108758_0", "node+edge", r"η0=0.25, L=6, D=512, seed=0"),
    JobSpec("13110655_0", "node+edge", r"η0=1, L=6, D=512, seed=0"),
    JobSpec("13113754", "node+edge", r"η0=2.5, L=6, D=512, seed=0"),
]

CAPEN_LLAMA_JOBS = [
    JobSpec("13127462_0", "node", r"η0=0.01, L=4, D=256, seed=0"),
    JobSpec("13127470_0", "node", r"η0=0.05, L=4, D=256, seed=0"),
    JobSpec("13127473_0", "node", r"η0=0.25, L=4, D=256, seed=0"),
    JobSpec("13127474_0", "node", r"η0=1, L=4, D=256, seed=0"),
    JobSpec("13127475_0", "node", r"η0=2.5, L=4, D=256, seed=0"),
    #
    JobSpec("13127498_0", "node", r"η0=0.01, L=6, D=512, seed=0"),
    JobSpec("13127499_0", "node", r"η0=0.05, L=6, D=512, seed=0"),
    JobSpec("13127505_0", "node", r"η0=0.25, L=6, D=512, seed=0"),
    JobSpec("13127506_0", "node", r"η0=1, L=6, D=512, seed=0"),
    JobSpec("13127509_0", "node", r"η0=2.5, L=6, D=512, seed=0"),
    #
    JobSpec("13127227_0", "node+edge", r"η0=0.01, L=4, D=256, seed=0"),
    JobSpec("13127235_0", "node+edge", r"η0=0.05, L=4, D=256, seed=0"),
    JobSpec("13127236_0", "node+edge", r"η0=0.25, L=4, D=256, seed=0"),
    JobSpec("13127237_0", "node+edge", r"η0=1, L=4, D=256, seed=0"),
    JobSpec("13127239_0", "node+edge", r"η0=2.5, L=4, D=256, seed=0"),
    #
    JobSpec("13127213_0", "node+edge", r"η0=0.01, L=6, D=512, seed=0"),
    JobSpec("13126479_0", "node+edge", r"η0=0.05, L=6, D=512, seed=0"),
    JobSpec("13126471_0", "node+edge", r"η0=0.25, L=6, D=512, seed=0"),
    JobSpec("13126535_0", "node+edge", r"η0=1, L=6, D=512, seed=0"),
    JobSpec("13127199_0", "node+edge", r"η0=2.5, L=6, D=512, seed=0"),
]

JOBS = CPEN_JOBS + CAPEN_LLAMA_JOBS

for spec in JOBS:
    print(f"{spec.job_id:14s}  {spec.role:10s}  {spec.note}")

study = TransferStudy.from_jobs(
    JOBS,
    d_ref=256,
    out_root=PASCAL_OUT_ROOT,
    slurm_dirs=PASCAL_SLURM_DIRS,
)
print("OUT_ROOT =", study.out_root)
display(study.status())
summary = study.pascal_summary()
display(summary)


In [ ]:
# Color = (L, D). Solid / circles = node. Dashed / squares = node+edge.
# Left panel = CPEN, right = CAPEN-Llama. Smaller than the old val/test pair.
study.plot_pascal_readout_suite()
study.plot_pascal_f1_curves()

wide = summary.pivot_table(
    index=["model_family", "depth", "width", "eta_0"],
    columns="role",
    values=["val_f1_best", "val_acc_best", "test_f1", "test_acc", "train_loss_min"],
)
display(wide)
